# 3c — TomTom Traffic

Downloads TomTom Traffic Flow PBF tiles, decodes the `traffic_level` float
(0.0 = blocked, 1.0 = free flow), classifies it into congestion levels, and
map-matches to OSM edges.

Both the raw `traffic_level` and the classified congestion are stored in
`tomtom_congestion_history` — so you can re-classify later without re-fetching.

Requires `TOMTOM_API_KEY` in `../.env`.

---

## How it works

TomTom's [Traffic Flow Tile API](https://developer.tomtom.com/traffic-api/documentation/traffic-flow/vector-flow-tiles)
returns PBF (Protocol Buffer) tiles in the same z/x/y scheme as Mapbox, using the
same MVT format. Each feature in the `Traffic flow` layer has:

| Property | Type | Description |
|---|---|---|
| `traffic_level` | float 0–1 | Ratio of current speed to free-flow speed |
| `road_closure` | bool | `true` if the road is physically closed |
| `road_type` | string | OSM-like road class |

**Classification thresholds** (applied to `traffic_level`):

| Condition | Level |
|---|---|
| `road_closure = true` | `severe` |
| `traffic_level < 0.40` | `severe` |
| `0.40 ≤ traffic_level < 0.60` | `heavy` |
| `0.60 ≤ traffic_level < 0.85` | `moderate` |
| `traffic_level ≥ 0.85` | `low` |

**Map matching** uses the same geometric approach as Mapbox:
25 m corridor + bearing filter (≤ 45°) + 40% overlap threshold.
The most severe segment wins. The raw `traffic_level` of the winning segment is
also stored so thresholds can be adjusted later.

In [ ]:
%pip install mercantile mapbox-vector-tile python-dotenv folium --quiet

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '..')
from scripts.tomtom_traffic import TomTomTraffic
from scripts.traffic_db import TrafficDB, CONGESTION_COLORS

load_dotenv(Path('../.env'), override=True)

# ── Configuration ─────────────────────────────────────────────────────
NAME      = 'sodermalm'
ZOOM      = 14
DB        = f'../db/{NAME}.duckdb'
BOUNDARY  = f'../boundaries/{NAME}.geojson'
TILES_DIR = Path('../tiles/tomtom')
OUT_FILE  = Path(f'../output/{NAME}_traffic_tomtom.geojson')
API_KEY   = os.environ.get('TOMTOM_API_KEY', '')
# ─────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError('TOMTOM_API_KEY not set in ../.env')
print(f'API key  : {API_KEY[:8]}...{API_KEY[-4:]}')
print(f'DuckDB   : {DB}')
print(f'Boundary : {BOUNDARY}')
print(f'Zoom     : {ZOOM}')

---
## Step 1 — Fetch tiles

Downloads TomTom Traffic Flow PBF tiles, decodes the `Traffic flow` layer,
classifies `traffic_level` into congestion labels, and clips to the boundary.

Tiles already on disk are **reused without re-downloading**.
The output GeoJSON is also cached — delete `OUT_FILE` to force a fresh fetch.

The decoded GeoDataFrame has columns:
- `geometry` — road segment LineString in WGS-84
- `congestion` — classified level string
- `traffic_level` — raw float (0.0–1.0), useful for custom thresholds
- `road_closure` — bool
- `road_type` — road class string

In [ ]:
%%time
traffic = TomTomTraffic(api_key=API_KEY, zoom=ZOOM)
gdf = traffic.fetch(BOUNDARY, TILES_DIR, OUT_FILE)

print(f'Segments      : {len(gdf):,}')
print(f'Congestion    : {gdf["congestion"].value_counts().to_dict()}')
if 'traffic_level' in gdf.columns:
    print(f'traffic_level : {gdf["traffic_level"].min():.2f} – {gdf["traffic_level"].max():.2f}')
display(gdf.head(3))

---
## Step 1b — Visualize raw TomTom segments (optional)

The tooltip shows `traffic_level` so you can see the raw float alongside
the classified level — useful for tuning thresholds.

In [ ]:
import folium
import geopandas as gpd

lines  = gdf[gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
bds    = lines.total_bounds
center = [(bds[1]+bds[3])/2, (bds[0]+bds[2])/2]

m = folium.Map(location=center, zoom_start=14, tiles='OpenStreetMap')
folium.GeoJson(
    lines.__geo_interface__,
    style_function=lambda feat: {
        'color': CONGESTION_COLORS.get(feat['properties'].get('congestion', 'no data'), '#cccccc'),
        'weight': 3,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['congestion', 'traffic_level', 'road_closure', 'road_type'],
        aliases=['Congestion', 'Traffic level (0=blocked, 1=free)', 'Closed?', 'Road type'],
    ),
).add_to(m)
m

---
## Step 2 — Map match to OSM edges

Same geometric algorithm as Mapbox: 25 m corridor, bearing ≤ 45°, ≥ 40% overlap.

Returns two dicts:
- `edge_cong` — edge_id → classified congestion string
- `traffic_levels` — edge_id → raw `traffic_level` float of the winning segment

Both are written to `tomtom_congestion_history`.

In [ ]:
%%time
edge_cong, traffic_levels = traffic.map_match(DB, gdf)

from collections import Counter
print(f'Edges matched    : {len(edge_cong):,}')
print(f'Distribution     : {Counter(edge_cong.values())}')
print(f'Traffic levels   : {len(traffic_levels)} edges with raw float stored')

---
## Step 3 — Write to DuckDB

`tomtom_congestion_history` stores both `congestion` (string) and `traffic_level` (float).
This lets you later re-classify with different thresholds:
```sql
SELECT edge_id,
       CASE WHEN traffic_level < 0.30 THEN 'severe'
            WHEN traffic_level < 0.55 THEN 'heavy'
            WHEN traffic_level < 0.80 THEN 'moderate'
            ELSE 'low' END AS recls
FROM tomtom_congestion_history
WHERE run_id = 1;
```

In [ ]:
%%time
with TrafficDB(DB, read_only=False) as db:
    run_id = db.write_congestion(
        edge_cong,
        source          = 'tomtom',
        zoom            = ZOOM,
        n_segments      = len(gdf),
        boundary_name   = NAME,
        traffic_levels  = traffic_levels,
    )
print(f'Written as run_id={run_id}  source=tomtom  zoom={ZOOM}')

---
## Step 4 — Inspect results

In [ ]:
%%time
with TrafficDB(DB) as db:
    print('=== History ===')
    display(db.get_history_index())
    print('\n=== Congestion summary (TomTom) ===')
    display(db.get_congestion_summary('tomtom'))

    edges = db.get_edges(source='tomtom')
    m = db.plot_edges(edges)
m

---
## Step 5 — Compare all three sources (optional)

Run this after notebooks 3a and 3b have also been run.
Each panel shows the latest snapshot for that source.

In [ ]:
with TrafficDB(DB) as db:
    db.plot_comparison(sources=['mapbox', 'google', 'tomtom'])